# Stage 22 — Controlled downstream classifier across selected gene panels

**Purpose:** use exactly the same logistic classifier and normalization on **five method-defined gene panels** per outer LOCO fold. This is a new, versioned secondary experiment; original Stage 5/7/8/9/10 fitted models and patient predictions are preserved.

Panels: LASSO, elastic net, Stability Selection (fixed cutoff=0.60), author-native StableMate significantly-stable-predictive genes, and author-native TENTACLES `getConsensus` genes. Features are selected from outer **training** data in the prior runs; their original methods/hyperparameters are not rerun or modified. A held-out patient is never used for gene selection or scaler fit. StableMate's native model predictions and TENTACLES's Stage 10 logistic predictions remain separately labelled.

In [ ]:
from pathlib import Path
import os, sys, json
from datetime import datetime, timezone
import pandas as pd

candidates = [os.environ.get("CMPS355_ROOT"), Path.cwd(), Path.cwd().parent,
              "/content/drive/MyDrive/CMPS355_Research"]
ROOT = next((Path(x).resolve() for x in candidates if x and
             (Path(x)/"data/validated/ibd_dataset_sample_aligned.zip").is_file() and
             (Path(x)/"src/cmps355").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Set CMPS355_ROOT to the project root containing data/validated and src/cmps355")
sys.path.insert(0,str(ROOT / "src"))
RUN_ID = os.environ.get("CMPS355_RUN_ID") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
print("PROJECT_ROOT:", ROOT)
print("RUN_ID:", RUN_ID)


In [ ]:
from cmps355.common_classifier_benchmark import main
RUN_DIR=ROOT/'runs/common_classifier'/RUN_ID
report=main(ROOT,RUN_DIR)
assert report['status']=='PASS_IDENTICAL_DOWNSTREAM_MODEL_5_METHODS_4_FOLDS'
print('CONTROLLED_COMPARISON_STATUS:',report['status'])
print('Patient predictions are PRIVATE:',RUN_DIR/'patient_predictions_PRIVATE.csv')

In [ ]:
per_fold=pd.read_csv(RUN_DIR/'cohort_metrics.csv')
macro=pd.read_csv(RUN_DIR/'macro_summary.csv')
display(macro.round(4))
display(per_fold[['heldout','method','n_genes','auroc','balanced_accuracy']].round(4))

In [ ]:
print('Comparison of new TENTACLES probabilities to earlier frozen Stage 10:')
display(pd.DataFrame.from_dict(report['stage10_reproduction'],orient='index'))
print('Input panel hashes are recorded in report.json for all five selectors.')

## Interpretation

This controls the **downstream learner** and `log2(1+CPM)` preprocessing, **not the feature selector itself**. Upstream tuning budgets and panel sizes differ. Four LOCO held-out cohorts are the reporting units, and three-cohort training sets overlap, so neither 20 rows nor six panel-pair overlaps are independent experiments.

TENTACLES Stage 10 is already essentially the same downstream model. Small probability differences upon refitting may occur due to software version, numeric precision or solver stopping tolerance; the exact deltas are preserved without overwriting either set of predictions. Per-patient results stay private. Report aggregate metrics descriptively, not as proof one method is statistically superior.